# Stress DSA via `lic_dsf.stress`

Load Input 6 shock sizes, run standard external B-tests (B1 GDP, B3–B6),
and inspect **public residual financing** (ext MLT / dom MLT / ST) under B1 GDP.

See `docs/08-stress-dsa.qmd`.

In [1]:
from __future__ import annotations

from pathlib import Path

import pandas as pd

from lic_dsf.output import stress_external_panel, stress_public_panel
from lic_dsf.load import (
    load_external_debt_inputs,
    load_input6_standard,
    load_input7_residual_params,
    load_instruments_from_workbook,
    load_lc_nr_instruments_from_workbook,
    load_macro_debt_inputs,
)
from lic_dsf.pv import (
    ExternalDebtBook,
    MacroDebtBook,
    PVPortfolio,
)
from lic_dsf.stress import (
    run_b1_gdp_external,
    run_b1_gdp_public,
    run_standard_external_stress,
)

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "demo":
    REPO_ROOT = REPO_ROOT.parent

WORKBOOK = REPO_ROOT / "data" / "lic-dsf-template-2025-08-12.xlsx"

pd.set_option("display.max_columns", 24)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

WORKBOOK

PosixPath('/home/sravan/excel-grapher/lic-dsf/data/lic-dsf-template-2025-08-12.xlsx')

In [2]:
instruments = load_instruments_from_workbook(
    WORKBOOK, include_zero_disbursement=True
)
lc_nr = load_lc_nr_instruments_from_workbook(
    WORKBOOK, include_zero_disbursement=True
)
portfolio = PVPortfolio(instruments=tuple(instruments) + tuple(lc_nr))
external = ExternalDebtBook(
    portfolio=portfolio, inputs=load_external_debt_inputs(WORKBOOK)
)
macro = MacroDebtBook(inputs=load_macro_debt_inputs(WORKBOOK), external=external)
params = load_input6_standard(WORKBOOK)
residual = load_input7_residual_params(WORKBOOK)

(
    params.gdp_shock_sd,
    params.threshold_rule,
    residual.external_mlt_share,
    residual.domestic_mlt_share,
    residual.domestic_st_share,
)

(1.0,
 'whichever_lower',
 0.4343925896763784,
 0.2275655436226405,
 0.33804186670098113)

In [3]:
b1 = run_b1_gdp_external(macro, external, params, residual)
years = [2024, 2025, 2026, 2027]
pd.DataFrame(
    {
        "baseline GDP": macro.gdp_usd().reindex(years),
        "B1 GDP": b1.macro.gdp_usd().reindex(years),
        "B1 PV/GDP": b1.pv_ppg_external_to_gdp().reindex(years),
        "B1 PV/exports": b1.pv_ppg_external_to_exports().reindex(years),
    }
)

,baseline GDP,B1 GDP,B1 PV/GDP,B1 PV/exports
2024,"45,464.0861","45,464.0861",44.8846,109.0647
2025,"49,226.3138","45,239.0552",46.9546,103.4689
2026,"52,863.2698","46,720.5870",46.6468,99.5562
2027,"56,415.2097","49,859.7934",45.4787,98.4192


In [4]:
stress_external_panel(b1).loc[:, years]

,2024,2025,2026,2027
PV of PPG external debt / GDP,44.8846,46.9546,46.6468,45.4787
PV of PPG external debt / exports,109.0647,103.4689,99.5562,98.4192
PV of PPG external debt / revenue,247.7666,237.3607,221.1705,213.7632
PPG debt service / exports,17.1681,16.5074,15.1705,16.2756
PPG debt service / revenue,39.0015,37.8686,33.7022,35.3500
External GFN (USD),"2,360.2673","2,091.5041","2,101.2626","2,600.8647"


In [5]:
results = run_standard_external_stress(macro, external, params, residual)
pd.DataFrame(
    {sid: book.pv_ppg_external_to_gdp().reindex(years) for sid, book in results.items()}
)

,B1_GDP,B3_Exports,B4_OtherFlows,B5_FX,B6_Combo
2024,44.8846,44.8846,44.8846,44.8846,44.8846
2025,46.9546,48.0971,46.7521,54.7005,51.7962
2026,46.6468,55.6099,48.1803,52.2604,57.9999
2027,45.4787,62.5539,46.7100,50.9516,61.1725


In [6]:
# Public B1: three-way residual fill (ext MLT / dom MLT / ST)
pub = run_b1_gdp_public(macro, external, params, residual)
fill = pub.resfin.fill
pd.DataFrame(
    {
        "ext MLT USD": fill.external_mlt_usd.reindex(years),
        "dom MLT LCU": fill.domestic_mlt_lcu.reindex(years),
        "dom ST LCU": fill.domestic_st_lcu.reindex(years),
        "public debt/GDP": pub.public_sector_debt_to_gdp().reindex(years),
        "PV public debt/GDP": pub.pv_public_debt_to_gdp().reindex(years),
    }
)

,ext MLT USD,dom MLT LCU,dom ST LCU,public debt/GDP,PV public debt/GDP
2024,0.0000,0.0000,0.0000,69.7520,67.1667
2025,284.2335,730.6457,"1,085.3526",72.3888,69.4333
2026,526.3381,"1,456.9670","2,164.2813",73.4734,69.8627
2027,711.5032,"2,084.2737","3,096.1268",72.8067,68.7502


In [7]:
stress_public_panel(pub).loc[:, years]

,2024,2025,2026,2027
Public sector debt / GDP,69.7520,72.3888,73.4734,72.8067
PV of public debt / GDP,67.1667,69.4333,69.8627,68.7502
Debt service / revenue+grants,126.3839,92.4878,87.7751,88.3005
Public GFN (LCU),"45,533.9567","39,639.0524","45,503.1524","52,486.9787"
